# Clay & Craft Studio — Data Generator
Generates SQL INSERT files and matching CSVs for:
- `ProductInstances` → `product_instances_inserts.sql` + `product_instances_export.csv`
- `Customers` → `customers_inserts.sql` + `customer_export.csv`
- `Orders` + `OrderItems` → `orders_inserts.sql` + `order_items_inserts.sql`

Run cells **top to bottom**. The Orders cell reads the CSVs written by the two cells above it.

**Reproducibility:** All randomness is controlled by `SEED`


## Shared Configuration
Run this first — every other cell imports from it!

In [3]:
import random
import csv
import string
from datetime import datetime, timedelta

SEED = 42
random.seed(SEED)

# ---------------
# SHARED HELPERS
# ---------------
def weighted_choice(options):
    """Pick a random value from [(value, weight), ...] pairs."""
    values, weights = zip(*options)
    return random.choices(values, weights=weights, k=1)[0]

def write_sql(filename, insert_stmt, records, col_order, batch_size=999):
    """
    Writes a list of dicts to a batched SQL INSERT file. col_order defines which dict keys to include and in what order.
    Values are quoted if they are strings, left alone if int/float.
    """
    def fmt(v):
        return f"'{v}'" if isinstance(v, str) else str(v)

    with open(filename, "w") as f:
        for i in range(0, len(records), batch_size):
            batch = records[i:i + batch_size]
            f.write(insert_stmt + " VALUES\n")
            for j, rec in enumerate(batch):
                vals = ", ".join(fmt(rec[c]) for c in col_order)
                suffix = "\n" if j == len(batch) - 1 else ",\n"
                f.write(f"({vals}){suffix}")
            f.write(";\n\n")

def write_csv(filename, records, col_order):
    """Write a list of dicts to a CSV file using the given column order as the header."""
    with open(filename, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=col_order, extrasaction="ignore")  # ← add this
        writer.writeheader()
        writer.writerows(records)

print(f"Shared config finished. SEED = {SEED}")

Shared config finished. SEED = 42


## 1 — Product Instance Generator
Simulates studio production day-by-day from Jan 2019 to Apr 2026.
Staff join progressively and volume, quality, and status all vary by era.

| Period | Date Range | Active Staff |
|--------|------------|--------------|
| 1 | Jan 2019 – Jun 2019 | Sophie |
| 2 | Jun 2019 – Sep 2019 | Sophie, Joan |
| 3 | Sep 2019 – Nov 2021 | Sophie, Joan, Milo |
| 4 | Nov 2021 – Jan 2024 | Sophie, Joan, Milo, Leila |
| 5 | Jan 2024 – Jan 2025 | Joan, Milo, Leila, Ben |
| 6 | Jan 2025 – Apr 2026 | Joan, Milo, Leila, Ben |

In [4]:
PI_SQL_FILE = "product_instances_inserts.sql"
PI_CSV_FILE = "product_instances_export.csv"

START_DATE = datetime(2019, 1, 1)
END_DATE = datetime(2026, 4, 1)
PRODUCT_IDS = [1, 2, 3, 4, 5, 6]
GLAZES = ["matte", "glossy", "speckled", "crackle"]

# Maps staff name to staff_id (matches Staff table)
STAFF = {"sophie": 1, "joan": 2, "milo": 3, "leila": 4, "ben": 5}

STAFF_BY_PERIOD = {
    1: ["sophie"],
    2: ["sophie", "joan"],
    3: ["sophie", "joan", "milo"],
    4: ["sophie", "joan", "milo", "leila"],
    5: ["joan", "milo", "leila", "ben"],
    6: ["joan", "milo", "leila", "ben"],
}

DAILY_VOLUME = {1: 3, 2: 6, 3: 9, 4: 12, 5: 12, 6: 15} # 3 × multiplier

# -------------------
# HELPER FUNCTIONS
# -------------------
def get_period(date):
    """Map a date to a studio production period (1–6) based on staff hire dates."""
    if date < datetime(2019, 6, 10): return 1
    if date < datetime(2019, 9, 5): return 2
    if date < datetime(2021, 11, 22): return 3
    if date < datetime(2024, 1, 1): return 4
    if date < datetime(2025, 1, 1): return 5
    return 6

def get_quality(staff, period):
    """Return a quality grade (A/B/C) weighted by the artist's skill level and tenure."""
    if staff == "sophie": return weighted_choice([("A", 98), ("B", 2)])
    if staff == "joan": return weighted_choice([("A", 95), ("B", 5)])
    if staff == "milo":
        if period >= 5: return weighted_choice([("B", 60), ("C", 40)]) # dips later
        return weighted_choice([("A", 60), ("B", 40)])
    if staff == "leila":
        if period < 4: return weighted_choice([("A", 60), ("B", 35), ("C", 5)]) # improves
        return weighted_choice([("A", 70), ("B", 30)])
    if staff == "ben": return weighted_choice([("B", 40), ("C", 60)])

def get_status(staff, period):
    """Return a status (available/sold/defective) based on the era and artist."""
    if period <= 3:
        return "defective" if random.random() < 0.01 else "sold" # early years mostly sold
    if period == 4:
        if staff == "leila": return weighted_choice([("sold", 80), ("defective", 20)])
        return "sold"
    if period == 5:
        if staff == "ben": return weighted_choice([("sold", 50), ("available", 25), ("defective", 25)])
        if staff == "leila": return weighted_choice([("sold", 70), ("available", 30)])
        return "sold"
    # period 6
    if staff == "milo": return weighted_choice([("defective", 40), ("available", 40), ("sold", 20)])
    return weighted_choice([("available", 60), ("sold", 35), ("defective", 5)])

# --------------
# GENERATE DATA
# --------------
records = [] # list of dicts, single source of truth for both SQL and CSV
instance_id = 1
current_date = START_DATE

while current_date <= END_DATE:
    period = get_period(current_date)

    for _ in range(DAILY_VOLUME[period]):
        staff = random.choice(STAFF_BY_PERIOD[period])

        records.append({
            "instance_id": instance_id,
            "product_id": random.choice(PRODUCT_IDS),
            "created_by": STAFF[staff],
            "date_created": str(current_date.date()),
            "glaze_type": random.choice(GLAZES),
            "quality_grade": get_quality(staff, period),
            "status": get_status(staff, period),
        })
        instance_id += 1

    current_date += timedelta(days=1)

# -----------------
# WRITE SQL + CSV
# -----------------
SQL_COLS = ["product_id", "created_by", "date_created", "glaze_type", "quality_grade", "status"]
CSV_COLS = ["instance_id", "date_created", "status"]

write_sql(PI_SQL_FILE, "INSERT INTO ProductInstances (product_id, created_by, date_created, glaze_type, quality_grade, status)", records, SQL_COLS)
write_csv(PI_CSV_FILE, records, CSV_COLS)

print(f"Generated {len(records):,} product instances")

Generated 28,755 product instances


## 2 — Customer Generator
Generates 5,000 fictional customers. Names are built from randomised syllable parts to avoid
accidentally matching real people. A 6-character random suffix on each email prevents duplicates.

In [5]:
CUST_SQL_FILE = "customers_inserts.sql"
CUST_CSV_FILE = "customer_export.csv"
TOTAL_CUSTOMERS = 5000

FIRST_PREFIXES = ["Al","Be","Ca","Da","El","Fa","Jo","Ka","Le","Ma","Na","Ol","Pa","Ra","Sa","Ta","Vi","Za"]
FIRST_SUFFIXES = ["den","son","ria","lin","mar","ton","sha","rel","vin","nor","lan","bel","ron","sel","mon"]
LAST_PREFIXES = ["Har","Wil","Tor","Fen","Dal","Mor","Kel","Nor","Bel","Cor","Len","Van","San","Dar","Pal"]
LAST_SUFFIXES = ["son","man","ford","ley","ton","well","berg","field","stone","wood","brook","hart","ridge"]
DOMAINS = ["example.com","mail.test","fakemail.org","demo.net"]

# ------------------
# HELPER FUNCTIONS
# -----------------
def generate_name():
    """Creates a random fictional first + last name from syllable parts."""
    first = random.choice(FIRST_PREFIXES) + random.choice(FIRST_SUFFIXES)
    last = random.choice(LAST_PREFIXES) + random.choice(LAST_SUFFIXES)
    return first, last

def generate_email(first, last):
    """Creates a fake email with a random 6-char suffix."""
    rand = "".join(random.choices(string.ascii_lowercase + string.digits, k=6))
    return f"{first.lower()}.{last.lower()}{rand}@{random.choice(DOMAINS)}"

# -----------------------------
# GENERATE UNIQUE CUSTOMERS
# -----------------------------
seen_names = set()
records = []
customer_id = 1

while len(records) < TOTAL_CUSTOMERS:
    first, last = generate_name()
    full_name = f"{first} {last}"

    if full_name in seen_names:
        continue

    seen_names.add(full_name)

    records.append({
        "customer_id": customer_id,
        "name": full_name,
        "email": generate_email(first, last),
    })
    customer_id += 1

# -----------------
# WRITE SQL + CSV
# -----------------
SQL_COLS = ["name", "email"]
CSV_COLS = ["customer_id", "name", "email"]

write_sql(CUST_SQL_FILE, "INSERT INTO Customers (name, email)", records, SQL_COLS)
write_csv(CUST_CSV_FILE, records, CSV_COLS)

print(f"Generated {len(records):,} unique customers")

Generated 5,000 unique customers


## 3 — Orders + Order Items Generator
Reads the CSVs written by cells 1 and 2 **run those first!**

Only `sold` product instances are ordered. Nearby instances (within 3 days) are grouped into the same order to simulate realistic multi-item purchases. The customer pool grows linearly over time so early orders only come from a small initial customer base.

In [6]:
ORDERS_SQL_FILE = "orders_inserts.sql"
ORDER_ITEMS_SQL_FILE = "order_items_inserts.sql"

# -----------------------------
# HELPER FUNCTIONS
# -----------------------------
def load_product_instances(filepath):
    """Load only 'sold' product instances from CSV, sorted chronologically."""
    instances = []
    with open(filepath) as f:
        for row in csv.DictReader(f):
            if row["status"] == "sold":
                instances.append({
                    "id":   int(row["instance_id"]),
                    "date": datetime.strptime(row["date_created"], "%Y-%m-%d"),
                })
    instances.sort(key=lambda x: x["date"])
    return instances

def load_customers(filepath):
    """Load customer IDs from CSV in insertion order."""
    with open(filepath) as f:
        return [int(row["customer_id"]) for row in csv.DictReader(f)]

def get_customer_pool(date, customers):
    """Return the customers available at a given date. Pool grows linearly from 5 customers at open to the full list by Apr 2026."""
    start = datetime(2019, 1, 1)
    end = datetime(2026, 4, 1)
    progress = (date - start).days / (end - start).days
    size = max(5, int(5 + (len(customers) - 5) * progress))
    return customers[:size]

# ------------
# LOAD DATA
# ------------
product_instances = load_product_instances(PI_CSV_FILE)
customers = load_customers(CUST_CSV_FILE)

# -----------------------------
# GENERATE ORDERS + ORDER ITEMS
# -----------------------------
order_records = []
order_item_records = []
order_id = 1
order_item_id = 1
i = 0

while i < len(product_instances):
    current = product_instances[i]
    order_date = current["date"]

    customer_id = random.choice(get_customer_pool(order_date, customers))
    group_size = random.randint(1, 3)

    # Collect up to group_size instances created within 3 days of this order date
    group = [current]
    j = i + 1
    while j < len(product_instances) and len(group) < group_size:
        if abs((product_instances[j]["date"] - order_date).days) <= 3:
            group.append(product_instances[j])
            j += 1
        else:
            break

    order_records.append({
        "order_id": order_id,
        "customer_id": customer_id,
        "order_date": str(order_date.date()),
        "qty": len(group),
    })

    for item in group:
        order_item_records.append({
            "order_item_id": order_item_id,
            "order_id": order_id,
            "product_instance_id": item["id"],
            "price_sold": round(random.uniform(15, 80), 2),
        })
        order_item_id += 1

    order_id += 1
    i += len(group)

# ------------
# WRITE SQL
# ------------
write_sql(ORDERS_SQL_FILE, "INSERT INTO Orders (customer_id, order_date, qty)", order_records, ["customer_id", "order_date", "qty"])
write_sql(ORDER_ITEMS_SQL_FILE, "INSERT INTO OrderItems (order_id, product_instance_id, price_sold)", order_item_records, ["order_id", "product_instance_id", "price_sold"])

print(f"Generated {len(order_records):,} orders")
print(f"Generated {len(order_item_records):,} order items")
print(f"Sold instances used: {len(product_instances):,}")

Generated 11,251 orders
Generated 22,618 order items
Sold instances used: 22,618
